# Baseline Comparison — BRACOL Dataset

So sánh **train from scratch** (không dùng pretrained weights) trên cùng điều kiện:
- Cùng dataset: BRACOL 1685 ảnh, split 70/15/15
- Cùng augmentation
- Cùng 50 epoch, cùng optimizer, cùng scheduler
- Cùng class weights

**3 baseline:**
1. ResNet-18 (train from scratch)
2. EfficientNet-B0 (train from scratch)
3. MobileViT-S (train from scratch)

**Lưu ý:** Vào `Runtime > Change runtime type > T4 GPU` trước khi chạy

In [ ]:
# ── Cell 1: Kiểm tra GPU ────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('KHÔNG CÓ GPU — vào Runtime > Change runtime type > T4 GPU')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

PyTorch: 2.10.0+cu128
GPU   : NVIDIA A100-SXM4-40GB
VRAM  : 42.4 GB
Device: cuda


In [ ]:
!pip install timm einops -q

import os, time, json
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import timm

print('Tất cả thư viện OK')
print('timm version:', timm.__version__)

Tất cả thư viện OK
timm version: 1.0.26


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    'data_dir':     '/content/drive/MyDrive/BRACoL',
    'save_dir':     '/content/drive/MyDrive/KLTN_baseline',

    'classes':      ['cercospora', 'healthy', 'miner', 'phoma', 'rust'],
    'num_classes':  5,
    'train_ratio':  0.70,
    'val_ratio':    0.15,
    'test_ratio':   0.15,

    # ── Training
    'image_size':   224,
    'batch_size':   32,
    'epochs':       50,
    'lr':           2e-4,
    'weight_decay': 1e-4,
    'warmup_epochs':5,
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)
print('Config OK | save_dir:', CONFIG['save_dir'])

Mounted at /content/drive
Config OK | save_dir: /content/drive/MyDrive/KLTN_baseline


In [ ]:
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std  = (0.229, 0.224, 0.225)

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.2, 0.2, 0.15, 0.05),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]),
    'val_test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]),
}

def get_dataloaders():
    full_dataset = datasets.ImageFolder(root=CONFIG['data_dir'])
    labels       = full_dataset.targets

    train_idx, temp_idx = train_test_split(
        np.arange(len(labels)),
        test_size=(1 - CONFIG['train_ratio']),
        random_state=42, stratify=labels
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5,
        random_state=42, stratify=[labels[i] for i in temp_idx]
    )

    train_ds = Subset(datasets.ImageFolder(
        root=CONFIG['data_dir'], transform=data_transforms['train']),  train_idx)
    val_ds   = Subset(datasets.ImageFolder(
        root=CONFIG['data_dir'], transform=data_transforms['val_test']), val_idx)
    test_ds  = Subset(datasets.ImageFolder(
        root=CONFIG['data_dir'], transform=data_transforms['val_test']), test_idx)

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)

    # Class weights từ train split
    train_labels = [labels[i] for i in train_idx]
    counts  = np.bincount(train_labels,
                          minlength=CONFIG['num_classes']).astype(float)
    weights = torch.tensor(counts.max() / counts, dtype=torch.float32)

    print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
    print('Class weights:', {
        c: round(w, 2) for c, w in zip(CONFIG['classes'], weights.tolist())
    })
    return train_loader, val_loader, test_loader, weights

train_loader, val_loader, test_loader, class_weights = get_dataloaders()

Train: 1179 | Val: 253 | Test: 253
Class weights: {'cercospora': 3.61, 'healthy': 1.96, 'miner': 1.37, 'phoma': 1.53, 'rust': 1.0}


In [ ]:
def build_resnet18(num_classes):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(512, num_classes)
    return model


def build_efficientnet_b0(num_classes):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_mobilevit_s(num_classes):
    model = timm.create_model(
        'mobilevit_s',
        pretrained=False,
        num_classes=num_classes
    )
    return model


dummy = torch.randn(2, 3, 224, 224)
for name, fn in [
    ('ResNet-18',       build_resnet18),
    ('EfficientNet-B0', build_efficientnet_b0),
    ('MobileViT-S',     build_mobilevit_s),
]:
    m = fn(CONFIG['num_classes'])
    p = sum(x.numel() for x in m.parameters() if x.requires_grad)
    out = m(dummy)
    print(f'{name:20s} | Params: {p/1e6:.2f}M | Output: {out.shape}')

ResNet-18            | Params: 11.18M | Output: torch.Size([2, 5])
EfficientNet-B0      | Params: 4.01M | Output: torch.Size([2, 5])
MobileViT-S          | Params: 4.94M | Output: torch.Size([2, 5])


In [ ]:
def build_optimizer_scheduler(model, epochs, lr, weight_decay, warmup_epochs):
    optimizer = optim.AdamW(model.parameters(),
                            lr=lr, weight_decay=weight_decay)
    warmup = LinearLR(optimizer,
                      start_factor=0.1, end_factor=1.0,
                      total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer,
                               T_max=epochs - warmup_epochs,
                               eta_min=1e-6)
    scheduler = SequentialLR(optimizer,
                             schedulers=[warmup, cosine],
                             milestones=[warmup_epochs])
    return optimizer, scheduler


def run_epoch(model, loader, criterion, optimizer=None, training=True):
    model.train() if training else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            preds = out.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.detach().cpu().tolist())
            all_labels.extend(labels.detach().cpu().tolist())

    avg_loss = total_loss / total
    acc      = correct / total
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1


def train_model(name, model, epochs=None):
    if epochs is None:
        epochs = CONFIG['epochs']

    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        label_smoothing=0.1
    )
    optimizer, scheduler = build_optimizer_scheduler(
        model, epochs,
        CONFIG['lr'], CONFIG['weight_decay'], CONFIG['warmup_epochs']
    )

    history      = {'train_loss':[], 'train_acc':[], 'train_f1':[],
                    'val_loss':[],   'val_acc':[],   'val_f1':[]}
    best_val_f1  = 0.0
    ckpt_path    = os.path.join(CONFIG['save_dir'], f'best_{name}.pth')

    print(f'\n{"="*70}')
    print(f'  Training: {name}')
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {params/1e6:.2f}M | Epochs: {epochs} | LR: {CONFIG["lr"]}')
    print(f'{"="*70}')
    print(f"{'Ep':>4} | {'LR':>9} | {'TrLoss':>7} | {'TrAcc':>6} | "
          f"{'ValLoss':>7} | {'ValAcc':>6} | {'ValF1':>6} | Time")
    print('-' * 75)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = run_epoch(
            model, train_loader, criterion, optimizer, training=True)
        va_loss, va_acc, va_f1 = run_epoch(
            model, val_loader,   criterion, training=False)
        scheduler.step()
        cur_lr = scheduler.get_last_lr()[0]

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['train_f1'].append(tr_f1)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        history['val_f1'].append(va_f1)

        flag = ''
        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            torch.save(model.state_dict(), ckpt_path)
            flag = ' ←'

        print(f"{epoch:>4} | {cur_lr:>9.2e} | {tr_loss:>7.4f} | "
              f"{tr_acc*100:>5.1f}% | {va_loss:>7.4f} | "
              f"{va_acc*100:>5.1f}% | {va_f1:>6.4f}{flag} | "
              f"{time.time()-t0:.0f}s")

    print(f'\nBest val F1: {best_val_f1:.4f}')

    # Lưu history
    with open(os.path.join(CONFIG['save_dir'],
                           f'history_{name}.json'), 'w') as f:
        json.dump(history, f)

    return history, ckpt_path, model


def evaluate_model(name, model, ckpt_path):
    """Load best checkpoint và evaluate trên test set."""
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE), label_smoothing=0.1)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    acc  = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    report = classification_report(
        all_labels, all_preds,
        target_names=CONFIG['classes'], zero_division=0
    )

    print(f'\n{"-"*50}')
    print(f'  {name} — Test Results')
    print(f'  Accuracy : {acc*100:.2f}%')
    print(f'  Macro F1 : {f1:.4f}')
    print(f'{"-"*50}')
    print(report)
    return acc, f1


print('Utilities OK')

Utilities OK


In [ ]:
# ── Cell 7: Train ResNet-18 ──────────────────────────────────────
resnet18 = build_resnet18(CONFIG['num_classes'])
hist_r18, ckpt_r18, resnet18 = train_model('resnet18', resnet18)


  Training: resnet18
  Params: 11.18M | Epochs: 50 | LR: 0.0002
  Ep |        LR |  TrLoss |  TrAcc | ValLoss | ValAcc |  ValF1 | Time
---------------------------------------------------------------------------
   1 |  5.60e-05 |  1.6146 |  27.7% |  1.6035 |  35.2% | 0.2336 ← | 518s
   2 |  9.20e-05 |  1.4948 |  41.6% |  1.4777 |  45.1% | 0.3576 ← | 22s
   3 |  1.28e-04 |  1.4003 |  45.8% |  1.7682 |  29.6% | 0.2330 | 22s
   4 |  1.64e-04 |  1.3606 |  48.2% |  1.5606 |  42.3% | 0.3258 | 20s
   5 |  2.00e-04 |  1.3574 |  48.5% |  1.3999 |  55.3% | 0.4883 ← | 21s
   6 |  2.00e-04 |  1.3658 |  46.7% |  1.2968 |  36.8% | 0.3716 | 22s
   7 |  1.99e-04 |  1.3146 |  49.4% |  1.2265 |  60.1% | 0.5688 ← | 21s
   8 |  1.98e-04 |  1.2552 |  58.5% |  1.2393 |  64.8% | 0.5273 | 22s
   9 |  1.96e-04 |  1.1767 |  59.6% |  1.3463 |  62.5% | 0.4664 | 20s
  10 |  1.94e-04 |  1.1296 |  63.3% |  1.3302 |  65.6% | 0.5561 | 21s
  11 |  1.91e-04 |  1.0947 |  66.4% |  1.2435 |  68.4% | 0.6014 ← | 21s
  12 | 

In [ ]:
# ── Cell 8: Train EfficientNet-B0 ───────────────────────────────
effnet_b0 = build_efficientnet_b0(CONFIG['num_classes'])
hist_eff, ckpt_eff, effnet_b0 = train_model('efficientnet_b0', effnet_b0)


  Training: efficientnet_b0
  Params: 4.01M | Epochs: 50 | LR: 0.0002
  Ep |        LR |  TrLoss |  TrAcc | ValLoss | ValAcc |  ValF1 | Time
---------------------------------------------------------------------------
   1 |  5.60e-05 |  1.6601 |  24.5% |  1.6455 |  22.9% | 0.0746 ← | 21s
   2 |  9.20e-05 |  1.6530 |  23.1% |  1.6507 |  16.2% | 0.0558 | 21s
   3 |  1.28e-04 |  1.6526 |  22.8% |  1.6269 |   9.1% | 0.0396 | 20s
   4 |  1.64e-04 |  1.6472 |  20.2% |  1.6193 |  11.1% | 0.0885 ← | 21s
   5 |  2.00e-04 |  1.6053 |  25.1% |  1.6796 |  23.7% | 0.1849 ← | 21s
   6 |  2.00e-04 |  1.5661 |  34.3% |  1.5316 |  40.3% | 0.3509 ← | 21s
   7 |  1.99e-04 |  1.5038 |  38.9% |  1.5445 |  36.4% | 0.2768 | 21s
   8 |  1.98e-04 |  1.4681 |  43.3% |  1.4378 |  51.8% | 0.3685 ← | 21s
   9 |  1.96e-04 |  1.4443 |  42.5% |  1.3749 |  47.8% | 0.4169 ← | 21s
  10 |  1.94e-04 |  1.4247 |  40.4% |  1.8746 |  19.8% | 0.1740 | 21s
  11 |  1.91e-04 |  1.3971 |  41.8% |  1.4721 |  41.1% | 0.3899 | 21s


In [ ]:
# # ── Cell 9: Train MobileViT-S ────────────────────────────────────
mobilevit = build_mobilevit_s(CONFIG['num_classes'])
hist_mvit, ckpt_mvit, mobilevit = train_model('mobilevit_s', mobilevit)


  Training: mobilevit_s
  Params: 4.94M | Epochs: 50 | LR: 0.0002
  Ep |        LR |  TrLoss |  TrAcc | ValLoss | ValAcc |  ValF1 | Time
---------------------------------------------------------------------------
   1 |  5.60e-05 |  1.6379 |  13.0% |  1.6341 |  16.6% | 0.0632 ← | 21s
   2 |  9.20e-05 |  1.5928 |  22.6% |  1.4698 |  35.2% | 0.2620 ← | 21s
   3 |  1.28e-04 |  1.4886 |  39.1% |  1.5581 |  29.2% | 0.2319 | 21s
   4 |  1.64e-04 |  1.4177 |  46.6% |  1.4931 |  37.5% | 0.3060 ← | 21s
   5 |  2.00e-04 |  1.3709 |  51.1% |  1.3867 |  45.1% | 0.3898 ← | 21s
   6 |  2.00e-04 |  1.3530 |  50.4% |  1.4027 |  39.1% | 0.3460 | 21s
   7 |  1.99e-04 |  1.3067 |  52.1% |  1.3594 |  48.6% | 0.4805 ← | 21s
   8 |  1.98e-04 |  1.3004 |  52.7% |  1.3861 |  51.8% | 0.4531 | 21s
   9 |  1.96e-04 |  1.2602 |  56.7% |  1.4065 |  49.8% | 0.4834 ← | 21s
  10 |  1.94e-04 |  1.2446 |  58.5% |  1.3153 |  54.9% | 0.4912 ← | 21s
  11 |  1.91e-04 |  1.2158 |  63.5% |  1.2025 |  62.8% | 0.5801 ← | 21s


In [ ]:
print('\n' + '='*60)
print('  TEST SET EVALUATION — TẤT CẢ BASELINE')
print('='*60)

results = {}

for name, model, ckpt in [
    ('ResNet-18',       resnet18,  ckpt_r18),
    ('EfficientNet-B0', effnet_b0, ckpt_eff),
    ('MobileViT-S',     mobilevit, ckpt_mvit),
]:
    acc, f1 = evaluate_model(name, model, ckpt)
    results[name] = {'acc': acc, 'f1': f1}


  TEST SET EVALUATION — TẤT CẢ BASELINE

--------------------------------------------------
  ResNet-18 — Test Results
  Accuracy : 88.54%
  Macro F1 : 0.8618
--------------------------------------------------
              precision    recall  f1-score   support

  cercospora       0.75      0.68      0.71        22
     healthy       0.83      0.95      0.89        41
       miner       0.85      0.88      0.86        58
       phoma       0.98      0.87      0.92        53
        rust       0.92      0.92      0.92        79

    accuracy                           0.89       253
   macro avg       0.87      0.86      0.86       253
weighted avg       0.89      0.89      0.89       253


--------------------------------------------------
  EfficientNet-B0 — Test Results
  Accuracy : 79.84%
  Macro F1 : 0.7760
--------------------------------------------------
              precision    recall  f1-score   support

  cercospora       0.44      0.77      0.56        22
     healthy   

In [ ]:
# # ── Cell 11: Bảng so sánh tổng hợp ──────────────────────────────
# results['DepthCoAt-18 Plus'] = {'acc': 0.7984, 'f1': 0.7399}

params_map = {
    'ResNet-18':           11.7,
    'EfficientNet-B0':      5.3,
    'MobileViT-S':          5.6,
    'DepthCoAt-18 Plus':    6.32,
}

print('\n' + '='*65)
print(f"  {'Model':<22} | {'Params':>7} | {'Test Acc':>8} | {'Macro F1':>8}")
print('  ' + '-'*60)
for name, res in sorted(results.items(),
                         key=lambda x: x[1]['f1'], reverse=True):
    marker = ' ◄ (đề xuất)' if name == 'DepthCoAt-18 Plus' else ''
    print(f"  {name:<22} | {params_map[name]:>5.2f}M | "
          f"{res['acc']*100:>7.2f}% | {res['f1']:>8.4f}{marker}")
print('='*65)

# Lưu kết quả ra file
with open(os.path.join(CONFIG['save_dir'], 'comparison_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print('\nĐã lưu kết quả:', os.path.join(CONFIG['save_dir'], 'comparison_results.json'))

In [1]:
# # ── Cell 12: Visualize so sánh ───────────────────────────────────
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
# model_names = ['ResNet-18', 'EfficientNet-B0', 'MobileViT-S', 'DepthCoAt-18 Plus']
# histories   = [hist_r18, hist_eff, hist_mvit, None]  # None cho DepthCoAt (đã train trước)

# # Plot Val Accuracy curve
# ax = axes[0]
# for i, (name, hist) in enumerate(zip(model_names[:3], [hist_r18, hist_eff, hist_mvit])):
#     ep = range(1, len(hist['val_acc']) + 1)
#     ax.plot(ep, [a*100 for a in hist['val_acc']],
#             color=colors[i], label=name, linewidth=1.5)
# ax.axhline(y=79.84, color=colors[3], linestyle='--',
#            linewidth=2, label='DepthCoAt-18 Plus (best)')
# ax.set_title('Val Accuracy (%)'); ax.set_xlabel('Epoch')
# ax.legend(fontsize=8); ax.grid(alpha=0.3)

# # Plot Val F1 curve
# ax = axes[1]
# for i, (name, hist) in enumerate(zip(model_names[:3], [hist_r18, hist_eff, hist_mvit])):
#     ep = range(1, len(hist['val_f1']) + 1)
#     ax.plot(ep, hist['val_f1'],
#             color=colors[i], label=name, linewidth=1.5)
# ax.axhline(y=0.7399, color=colors[3], linestyle='--',
#            linewidth=2, label='DepthCoAt-18 Plus (best)')
# ax.set_title('Val Macro-F1'); ax.set_xlabel('Epoch')
# ax.legend(fontsize=8); ax.grid(alpha=0.3)

# # Bar chart so sánh test F1
# ax = axes[2]
# names = list(results.keys())
# f1s   = [results[n]['f1'] for n in names]
# bars  = ax.bar(names, f1s, color=colors[:len(names)], width=0.5)
# ax.set_title('Test Macro-F1 (so sánh cuối)')
# ax.set_ylabel('Macro F1')
# ax.set_ylim(0, 1.0)
# ax.tick_params(axis='x', rotation=15)
# for bar, v in zip(bars, f1s):
#     ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
#             f'{v:.4f}', ha='center', va='bottom', fontsize=9)
# ax.grid(axis='y', alpha=0.3)

# plt.suptitle('Baseline Comparison — BRACOL (Train from Scratch)',
#              fontsize=13, fontweight='bold')
# plt.tight_layout()
# save_path = os.path.join(CONFIG['save_dir'], 'baseline_comparison.png')
# plt.savefig(save_path, dpi=150, bbox_inches='tight')
# plt.show()
# print('Saved:', save_path)



**Lưu ý cercospora:**
- Class này chỉ có 22 ảnh test